# RNN Classifier with LSTM Trained on Own Dataset (IMDB)

Example notebook showing how to use an own CSV text dataset for training a simple RNN for sentiment classification (here: a binary classification problem with two labels, positive and negative) using LSTM (Long Short Term Memory) cells.

In [ ]:


import re
import time
import random
from collections import Counter

import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pad_sequence

torch.backends.cudnn.deterministic = True

## General Settings

In [ ]:
RANDOM_SEED = 123
torch.manual_seed(RANDOM_SEED)

VOCABULARY_SIZE = 20000
LEARNING_RATE = 0.005
BATCH_SIZE = 128
NUM_EPOCHS = 15
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_CLASSES = 2

## Download Dataset

The following cells will download the IMDB movie review dataset (http://ai.stanford.edu/~amaas/data/sentiment/) for positive-negative sentiment classification in as CSV-formatted file:

In [ ]:
import os
import gzip
import shutil
import urllib.request


URL = ('https://github.com/rasbt/python-machine-learning-book-3rd-edition/'
       'raw/master/ch08/movie_data.csv.gz')

if not os.path.exists('movie_data.csv'):
    urllib.request.urlretrieve(URL, 'movie_data.csv.gz')
    with gzip.open('movie_data.csv.gz', 'rb') as f_in, \
            open('movie_data.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove('movie_data.csv.gz')

Check that the dataset looks okay:

In [ ]:
df = pd.read_csv('movie_data.csv')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


## Prepare Dataset

The original version of this notebook used `torchtext.legacy` (`Field`, `TabularDataset`, `BucketIterator`), which was removed in torchtext 0.12; torchtext itself is no longer maintained. The cells below rebuild the same pipeline with plain Python and `torch.utils.data`, and use a simple regex tokenizer in place of spaCy.

Define the tokenizer:

In [ ]:
def tokenize(text):
    # Splits text into words and individual punctuation marks, similar to
    # the spaCy tokenizer used originally (case is preserved, as before)
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text)


print(tokenize("This movie wasn't bad... I'd watch it again!"))

['This', 'movie', "wasn't", 'bad', '.', '.', '.', "I'd", 'watch', 'it', 'again', '!']


## Split Dataset into Train/Validation/Test

Shuffle and split the dataset into training (68%), validation (12%), and test (20%) partitions:

In [ ]:
df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

num_test = int(0.2 * len(df))
test_df, train_df = df.iloc[:num_test], df.iloc[num_test:]

num_valid = int(0.15 * len(train_df))
valid_df, train_df = train_df.iloc[:num_valid], train_df.iloc[num_valid:]

print(f'Num Train: {len(train_df)}')
print(f'Num Validation: {len(valid_df)}')
print(f'Num Test: {len(test_df)}')

Num Train: 34000
Num Validation: 6000
Num Test: 10000


Tokenize the reviews:

In [ ]:
train_tokens = [tokenize(text) for text in train_df['review']]
valid_tokens = [tokenize(text) for text in valid_df['review']]
test_tokens = [tokenize(text) for text in test_df['review']]

print(train_tokens[0][:30], train_df['sentiment'].iloc[0])

['I', 'almost', 'burst', 'into', 'tears', 'watching', 'this', 'movie', '.', 'Not', 'from', 'laughing', 'but', 'from', 'the', 'memories', 'of', 'a', 'great', 'Rodney', 'Dangerfield', 'movie', '.', 'Candyshack', 'was', 'his', 'first', 'and', 'stole', 'the'] 0


## Build Vocabulary

Build the vocabulary based on the top "VOCABULARY_SIZE" words:

In [ ]:
word_counts = Counter(token for tokens in train_tokens for token in tokens)

# Indices 0 and 1 are reserved for the unknown-word and padding tokens
# (same layout as the legacy torchtext vocabulary)
itos = ['<unk>', '<pad>'] + [word for word, _ in word_counts.most_common(VOCABULARY_SIZE)]
stoi = {word: idx for idx, word in enumerate(itos)}
UNK_IDX, PAD_IDX = stoi['<unk>'], stoi['<pad>']

# The CSV already stores the labels as integers
class_mapping = {'negative': 0, 'positive': 1}

print(f'Vocabulary size: {len(itos)}')
print(f'Number of classes: {len(class_mapping)}')

Vocabulary size: 20002
Number of classes: 2


- 20,002 not 20,000 because of the `<unk>` and `<pad>` tokens
- PyTorch RNNs can deal with arbitrary lengths due to dynamic graphs, but padding is necessary for padding sequences to the same length in a given minibatch so we can store those in an array

**Look at most common words:**

In [ ]:
print(word_counts.most_common(20))

[('.', 443054), ('the', 391299), (',', 370308), ('a', 211004), ('and', 210815), ('of', 194261), ('to', 180133), ('/', 147079), ('is', 142445), ('>', 137996), ('<', 137917), ('br', 137800), ('in', 118749), ('I', 98387), ('that', 90695), ('it', 90251), ('"', 89966), ('-', 89952), ('this', 82033), ('was', 64725)]


**Tokens corresponding to the first 10 indices (0, 1, ..., 9):**

In [ ]:
print(itos[:10]) # itos = integer-to-string

['<unk>', '<pad>', '.', 'the', ',', 'a', 'and', 'of', 'to', '/']


**Converting a string to an integer:**

In [ ]:
print(stoi['the']) # stoi = string-to-integer

3


**Class labels:**

In [ ]:
print(class_mapping)

{'negative': 0, 'positive': 1}


**Class label count:**

In [ ]:
train_df['sentiment'].value_counts()

,count
sentiment,
1,17023
0,16977


## Define Data Loaders

`BucketBatchSampler` puts reviews of similar length into the same minibatch, and `collate_batch` pads each minibatch to its longest review:

In [ ]:
class IMDBDataset(Dataset):

    def __init__(self, token_lists, labels):
        self.texts = [torch.tensor([stoi.get(token, UNK_IDX) for token in tokens])
                      for tokens in token_lists]
        self.labels = torch.tensor(labels.values)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


class BucketBatchSampler(Sampler):
    """Groups examples of similar length into the same minibatch to
    minimize padding (what torchtext's BucketIterator did)."""

    def __init__(self, lengths, batch_size, shuffle=True, pool_factor=100):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.pool_size = batch_size * pool_factor

    def __iter__(self):
        indices = list(range(len(self.lengths)))
        if self.shuffle:
            random.shuffle(indices)
        batches = []
        # Sort by length within large pools so batches still vary across epochs
        for i in range(0, len(indices), self.pool_size):
            pool = sorted(indices[i:i + self.pool_size], key=lambda idx: self.lengths[idx])
            batches += [pool[j:j + self.batch_size]
                        for j in range(0, len(pool), self.batch_size)]
        if self.shuffle:
            random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size


def collate_batch(batch):
    texts, labels = zip(*batch)
    # pad_sequence returns [sentence length, batch size], matching the
    # default (seq_len, batch, features) layout of torch.nn.LSTM
    padded = pad_sequence(texts, padding_value=PAD_IDX)
    return padded, torch.stack(labels)


def make_loader(token_lists, labels, shuffle):
    dataset = IMDBDataset(token_lists, labels)
    sampler = BucketBatchSampler([len(text) for text in dataset.texts],
                                 BATCH_SIZE, shuffle=shuffle)
    return DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_batch)


random.seed(RANDOM_SEED)
train_loader = make_loader(train_tokens, train_df['sentiment'], shuffle=True)
valid_loader = make_loader(valid_tokens, valid_df['sentiment'], shuffle=False)
test_loader = make_loader(test_tokens, test_df['sentiment'], shuffle=False)

Testing the iterators (note that the number of rows depends on the longest document in the respective batch):

In [ ]:
for name, loader in [('Train', train_loader), ('Valid', valid_loader), ('Test', test_loader)]:
    text, labels = next(iter(loader))
    print(f'{name}:')
    print(f'Text matrix size: {text.size()}')
    print(f'Target vector size: {labels.size()}\n')

Train:
Text matrix size: torch.Size([64, 128])
Target vector size: torch.Size([128])

Valid:
Text matrix size: torch.Size([59, 128])
Target vector size: torch.Size([128])

Test:
Text matrix size: torch.Size([51, 128])
Target vector size: torch.Size([128])



## Model

In [ ]:
class RNN(torch.nn.Module):

    def __init__(self, input_dim, embedding_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = torch.nn.Embedding(input_dim, embedding_dim)
        #self.rnn = torch.nn.RNN(embedding_dim,
        #                        hidden_dim,
        #                        nonlinearity='relu')
        self.rnn = torch.nn.LSTM(embedding_dim,
                                 hidden_dim)

        self.fc = torch.nn.Linear(hidden_dim, output_dim)


    def forward(self, text):
        # text dim: [sentence length, batch size]

        embedded = self.embedding(text)
        # embedded dim: [sentence length, batch size, embedding dim]

        output, (hidden, cell) = self.rnn(embedded)
        # output dim: [sentence length, batch size, hidden dim]
        # hidden dim: [1, batch size, hidden dim]

        hidden.squeeze_(0)
        # hidden dim: [batch size, hidden dim]

        output = self.fc(hidden)
        return output

In [ ]:
torch.manual_seed(RANDOM_SEED)
model = RNN(input_dim=len(itos),
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            output_dim=NUM_CLASSES # could use 1 for binary classification
)

model = model.to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Training

In [ ]:
def compute_accuracy(model, data_loader, device):

    with torch.no_grad():

        correct_pred, num_examples = 0, 0

        for features, targets in data_loader:

            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)

            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100

In [ ]:
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    for batch_idx, (text, labels) in enumerate(train_loader):

        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        ### FORWARD AND BACK PROP
        logits = model(text)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()

        loss.backward()

        ### UPDATE MODEL PARAMETERS
        optimizer.step()

        ### LOGGING
        if not batch_idx % 50:
            print (f'Epoch: {epoch+1:03d}/{NUM_EPOCHS:03d} | '
                   f'Batch {batch_idx:03d}/{len(train_loader):03d} | '
                   f'Loss: {loss:.4f}')

    with torch.set_grad_enabled(False):
        print(f'training accuracy: '
              f'{compute_accuracy(model, train_loader, DEVICE):.2f}%'
              f'\nvalid accuracy: '
              f'{compute_accuracy(model, valid_loader, DEVICE):.2f}%')

    print(f'Time elapsed: {(time.time() - start_time)/60:.2f} min')

print(f'Total Training Time: {(time.time() - start_time)/60:.2f} min')
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 001/015 | Batch 000/266 | Loss: 0.6872
Epoch: 001/015 | Batch 050/266 | Loss: 0.6924
Epoch: 001/015 | Batch 100/266 | Loss: 0.6709
Epoch: 001/015 | Batch 150/266 | Loss: 0.6134
Epoch: 001/015 | Batch 200/266 | Loss: 0.6018
Epoch: 001/015 | Batch 250/266 | Loss: 0.5232
training accuracy: 78.66%
valid accuracy: 76.72%
Time elapsed: 0.25 min
Epoch: 002/015 | Batch 000/266 | Loss: 0.5114
Epoch: 002/015 | Batch 050/266 | Loss: 0.4333
Epoch: 002/015 | Batch 100/266 | Loss: 0.3938
Epoch: 002/015 | Batch 150/266 | Loss: 0.2631
Epoch: 002/015 | Batch 200/266 | Loss: 0.4227
Epoch: 002/015 | Batch 250/266 | Loss: 0.2937
training accuracy: 89.04%
valid accuracy: 85.52%
Time elapsed: 0.50 min
Epoch: 003/015 | Batch 000/266 | Loss: 0.2584
Epoch: 003/015 | Batch 050/266 | Loss: 0.2094
Epoch: 003/015 | Batch 100/266 | Loss: 0.2602
Epoch: 003/015 | Batch 150/266 | Loss: 0.1961
Epoch: 003/015 | Batch 200/266 | Loss: 0.2378
Epoch: 003/015 | Batch 250/266 | Loss: 0.1689
training accuracy: 88.99%
va

In [ ]:
def predict_sentiment(model, sentence):

    model.eval()

    with torch.no_grad():
        indexed = [stoi.get(token, UNK_IDX) for token in tokenize(sentence)]
        tensor = torch.LongTensor(indexed).to(DEVICE)
        tensor = tensor.unsqueeze(1)
        prediction = torch.nn.functional.softmax(model(tensor), dim=1)
    return prediction[0][class_mapping['positive']].item()

print('Probability positive:')
predict_sentiment(model, "This is such an awesome movie, I really love it!")

Probability positive:


0.9993566870689392

In [ ]:
print('Probability negative:')
1-predict_sentiment(model, "I really hate this movie. It is really bad and sucks!")

Probability negative:


0.9999912515850156